In [1]:
import pandas as pd
import ast
import re
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from itertools import chain

In [2]:
filepath = "../data/raw/vacancies_2026-04-14.csv"
DATE_STR = re.search(r'\d{4}-\d{2}-\d{2}', filepath).group()
df = pd.read_csv(f"../data/raw/vacancies_{DATE_STR}.csv")
print(df.shape)
df.head(3)

(1524, 17)


,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
0,132031395,700000.0,1200000.0,Алматы,between1And3,full,fullDay,"['ON_SITE', 'HYBRID']",['Дата-сайентист'],False,False,False,False,False,2026-04-10 16:02:39+03:00,Мы разрабатываем и развиваем продукт в сфере у...,ML Engineer / CV Engineer
1,131823118,9462.6,NaN,Астана,between1And3,full,remote,['REMOTE'],['Специалист технической поддержки'],False,False,True,False,True,2026-04-03 15:03:43+03:00,"Привет! Мы, ""Aspirity Solution"", студия разраб...",Customer Support Engineer
2,130648164,400000.0,800000.0,Астана,between1And3,full,fullDay,['ON_SITE'],['Дата-сайентист'],False,False,False,False,False,2026-03-20 15:49:17+03:00,Обязанности: • Разработка и улучшение NLP/LLM ...,ML Engineer (NLP / RAG)


In [3]:
df[(df['salary_from']<100_000) | (df['salary_to']<100_000)]

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
1,131823118,9462.60,NaN,Астана,between1And3,full,remote,['REMOTE'],['Специалист технической поддержки'],False,False,True,False,True,2026-04-03 15:03:43+03:00,"Привет! Мы, ""Aspirity Solution"", студия разраб...",Customer Support Engineer
64,131901002,9195.00,18390.00,Астана,between3And6,part,remote,['REMOTE'],"['BI-аналитик, аналитик данных']",False,False,True,False,False,2026-04-13 10:07:31+03:00,Обязанности: — Тестирование продуктовых фич с ...,Power BI Analyst (Product Testing)
763,131927036,4731.30,9462.60,Астана,moreThan6,project,remote,['REMOTE'],"['Программист, разработчик']",False,False,True,False,False,2026-04-07 17:17:05+03:00,Мы ищем опытного Full-Stack разработчика для у...,"AI-first Full Stack Developer (Node.js, React,..."
951,131632083,NaN,11081.60,Алматы,between3And6,project,fullDay,['ON_SITE'],"['Программист, разработчик']",False,False,True,True,False,2026-04-12 16:00:53+03:00,🔧 Программист промышленных роботов (ABB / Fanu...,Инженер-программист (АВВ/FANUC) с ВНЖ Финляндии
1435,131321300,2365.65,2365.65,Астана,between1And3,full,remote,['REMOTE'],['Тестировщик'],False,False,False,False,False,2026-04-11 06:05:53+03:00,"HDCart LIMITED – e-commerce компания, которая ...",QA Engineer (Tbilisi)


In [4]:
# removed non-it vacancies
df.drop(df[df['id'] == 131823118].index, inplace=True)

# we have some hours-paid vacancies, so we should convert them to monthly salary
df.loc[df['id'] == 131898390, ['salary_from', 'salary_to']] *= 4*5*4.33
df.loc[df['id'] == 131927036, ['salary_from', 'salary_to']] *= 6*5*4.33
df.loc[df['id'] == 131632083, ['salary_from', 'salary_to']] *= 8*5*4.33
df.loc[df['id'] == 131321300, ['salary_from', 'salary_to']] *= 8*5*4.33

In [5]:
df[(df['salary_from']<100_000) | (df['salary_to']<100_000)]

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
64,131901002,9195.0,18390.0,Астана,between3And6,part,remote,['REMOTE'],"['BI-аналитик, аналитик данных']",False,False,True,False,False,2026-04-13 10:07:31+03:00,Обязанности: — Тестирование продуктовых фич с ...,Power BI Analyst (Product Testing)


In [6]:
df['salary_mid'] = df[['salary_from', 'salary_to']].mean(axis=1)

In [7]:
print('Salary stats:')
print(df['salary_mid'].describe().apply(lambda x: f"{x:_.0f}"))

Salary stats:
count          486
mean       654_045
std        490_506
min         13_792
25%        350_000
50%        500_000
75%        750_000
max      3_311_910
Name: salary_mid, dtype: str


In [8]:
df[['name', 'area', 'salary_from', 'salary_to', 'salary_mid', 'professional_role']].sort_values('salary_mid').head(30)

,name,area,salary_from,salary_to,salary_mid,professional_role
64,Power BI Analyst (Product Testing),Астана,9195.0,18390.00,13792.50,"['BI-аналитик, аналитик данных']"
537,Junior разработчик,Алматы,100000.0,NaN,100000.00,"['Программист, разработчик']"
604,Стажер-программист 1С,Астана,100000.0,NaN,100000.00,"['Программист, разработчик']"
1052,Game Development International Internship,Алматы,118282.5,NaN,118282.50,['Гейм-дизайнер']
816,Game Development International Intern,Караганда,118282.5,NaN,118282.50,['Гейм-дизайнер']
1373,Системный администратор,Алматы,110000.0,150000.00,130000.00,['Системный администратор']
1378,Администратор-СММ в детском центре,Астана,100000.0,180000.00,140000.00,['Администратор']
1492,Project Management Trainee,Алматы,NaN,142885.26,142885.26,['Менеджер по персоналу']
1308,Помощник системного администратора,Астана,150000.0,NaN,150000.00,['Системный администратор']
1406,Администратор Bitrix24,Тараз,150000.0,NaN,150000.00,['Системный администратор']


In [9]:
df.head(2)

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name,salary_mid
0,132031395,700000.0,1200000.0,Алматы,between1And3,full,fullDay,"['ON_SITE', 'HYBRID']",['Дата-сайентист'],False,False,False,False,False,2026-04-10 16:02:39+03:00,Мы разрабатываем и развиваем продукт в сфере у...,ML Engineer / CV Engineer,950000.0
2,130648164,400000.0,800000.0,Астана,between1And3,full,fullDay,['ON_SITE'],['Дата-сайентист'],False,False,False,False,False,2026-03-20 15:49:17+03:00,Обязанности: • Разработка и улучшение NLP/LLM ...,ML Engineer (NLP / RAG),600000.0


In [10]:
df['work_format'] = df['work_format'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
df['professional_role'] = df['professional_role'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

df['role'] = df['professional_role'].apply(lambda x: x[0] if x else None)
df[['professional_role', 'role', 'work_format']].head(3)

,professional_role,role,work_format
0,[Дата-сайентист],Дата-сайентист,"[ON_SITE, HYBRID]"
2,[Дата-сайентист],Дата-сайентист,[ON_SITE]
3,[Дата-сайентист],Дата-сайентист,[REMOTE]


In [11]:
df[['professional_role', 'role', 'work_format']]

,professional_role,role,work_format
0,[Дата-сайентист],Дата-сайентист,"[ON_SITE, HYBRID]"
2,[Дата-сайентист],Дата-сайентист,[ON_SITE]
3,[Дата-сайентист],Дата-сайентист,[REMOTE]
4,[Дата-сайентист],Дата-сайентист,[ON_SITE]
5,[Дата-сайентист],Дата-сайентист,[ON_SITE]
...,...,...,...
1519,[DevOps-инженер],DevOps-инженер,[ON_SITE]
1520,[Администратор],Администратор,[REMOTE]
1521,[Системный администратор],Системный администратор,[ON_SITE]
1522,[Системный администратор],Системный администратор,[ON_SITE]


In [12]:
df['role'].unique()

<StringArray>
[                                                'Дата-сайентист',
                                                    'Инженер ПНР',
                                       'Программист, разработчик',
                                     'Технический директор (CTO)',
                                           'Продуктовый аналитик',
                                                 'DevOps-инженер',
                                   'BI-аналитик, аналитик данных',
                                                     'Архитектор',
                                                'Бизнес-аналитик',
                                                         'Другое',
                                            'Маркетолог-аналитик',
                                             'Системный аналитик',
                             'Менеджер по компенсациям и льготам',
                                  'Руководитель отдела аналитики',
                    'Менеджер по маркетингу, инт

In [13]:
# IT_ROLES is a determined by hh.kz list of professional_roles, here we just pick up only it related roles
IT_ROLES = [
    'Дата-сайентист',
    'Программист, разработчик',
    'Продуктовый аналитик',
    'DevOps-инженер',
    'BI-аналитик, аналитик данных',
    'Архитектор',
    'Бизнес-аналитик',
    'Другое',
    'Системный аналитик',
    'Системный администратор',
    'Аналитик',
    'Тестировщик',
    'Руководитель группы разработки',
    'Специалист по информационной безопасности',
    'Гейм-дизайнер',
    'Технический директор (CTO)',
    'Системный инженер',
]

# Same list as in first notebook, but with clarification - what kind of analysts we need
# and \b adds word boundaries to avoid false matches like "development" != "developer"
# engineer is specified to avoid false matches like "Pre-Sales Engineer"
IT_ROLE_KEYWORDS = [
    r"\bразработчик\b", r"\bdeveloper\b",
    r"\bsoftware engineer\b", r"\bdata engineer\b", r"\bsystems engineer\b",
    r"\barchitect\b", r"\bархитектор\b",
    r"\bios\b", r"\bandroid\b", r"\bfrontend\b", r"\bbackend\b", r"\bfullstack\b",
    r"\bdevops\b", r"\bmlops\b", r"\bqa\b", r"\bтестировщик\b", r"\bадминистратор\b",
    r"\bfullstack\b", r"\bsecurity engineer\b", r"\bdba\b",
    r"\bпрограммист\b", r"\bdata scientist\b", r"\bmachine learning\b",
    r"\bсистемный аналитик\b", r"\bпродуктовый аналитик\b", r"\bbi-аналитик\b",
    r"\bаналитик данных\b", r"\bdata analyst\b", r"\bбизнес-аналитик\b",
    r"\b1с\b", r"database\s+engineer", r"soc\b", r"information\s+security", r"automation\s+engineer", r"ux\s+engineer",
    r"software delivery", r"ict\b", r"devops", r"system integration",
]

df_cleaned = df[df['role'].isin(IT_ROLES)]
other = df_cleaned['role'] == 'Другое'
if_it_name = df_cleaned['name'].str.lower().str.contains('|'.join(IT_ROLE_KEYWORDS), na=False, regex=True)
df_cleaned = df_cleaned[~other | if_it_name].copy()

print(f'before deleting {len(df)} rows\nafter deleting:')
print(len(df_cleaned))

before deleting 1523 rows
after deleting:
1177


In [14]:
print(df_cleaned[df_cleaned['role'] == 'Другое'])

             id  salary_from  salary_to       area    experience employment  \
66    131898390          NaN        NaN     Астана  between3And6       full   
261   131992717          NaN        NaN     Алматы  between1And3       full   
355   131255736          NaN        NaN     Алматы  between3And6       full   
395   130572281          NaN   700000.0     Алматы  between3And6       full   
448   132022744          NaN        NaN     Астана  between3And6       full   
455   132022820          NaN        NaN     Астана  between3And6       full   
490   131918935          NaN        NaN     Алматы  between3And6       full   
522   131957279     230000.0        NaN  Караганда  between1And3       full   
530   131892702          NaN        NaN     Атырау  between3And6       full   
575   132010747          NaN        NaN     Алматы  between1And3       full   
633   132052137     500000.0  1000000.0     Алматы  between3And6    project   
680   131744265          NaN        NaN     Астана  

In [15]:
# non-it vacancy 
df_cleaned.drop(df_cleaned[df_cleaned['id'] == 131892702].index, inplace=True)

In [16]:
df_labeled = df_cleaned[df_cleaned['salary_mid'].notna()].copy()
df_unlabeled = df_cleaned[df_cleaned['salary_mid'].isna()].copy()

print(f"Labeled data have {len(df_labeled)} vacancies")
print(f"Unlabeled data have {len(df_unlabeled)} vacancies")

Labeled data have 367 vacancies
Unlabeled data have 809 vacancies


In [17]:
df_cleaned.to_csv(f'../data/processed/vacancies_clean_{DATE_STR}.csv', index=False)

In [18]:
total = len(df_cleaned)
pct = len(df_labeled) / total * 100

print(f"df_cleaned: {df_cleaned.shape}")
print(f"df_labeled: {df_labeled.shape}  ({pct:.1f}% of total)")
print(f"df_unlabeled: {df_unlabeled.shape}")

df_cleaned: (1176, 19)
df_labeled: (367, 19)  (31.2% of total)
df_unlabeled: (809, 19)


In [19]:
SKILL_ALIAS = {
    # Python
    "python": "Python", "python3": "Python", "python 3": "Python",
    # JavaScript
    "js": "JavaScript", "javascript": "JavaScript", "es6": "JavaScript", "es2015": "JavaScript",
    # TypeScript
    "ts": "TypeScript", "typescript": "TypeScript",
    # Java
    "java": "Java",
    # Go
    "go": "Go", "golang": "Go",
    # Kotlin
    "kotlin": "Kotlin",
    # Swift
    "swift": "Swift",
    # C++
    "c++": "C++", "cpp": "C++",
    # C#
    "c#": "C#", "csharp": "C#", "c sharp": "C#",
    # PHP
    "php": "PHP",
    # Ruby
    "ruby": "Ruby",
    # Scala
    "scala": "Scala",
    # Frameworks
    "django": "Django",
    "fastapi": "FastAPI", "fast api": "FastAPI",
    "flask": "Flask",
    "react": "React", "reactjs": "React", "react.js": "React",
    "vue": "Vue", "vuejs": "Vue", "vue.js": "Vue",
    "angular": "Angular", "angularjs": "Angular",
    "spring": "Spring", "spring boot": "Spring",
    ".net": ".NET", "dotnet": ".NET", "dot net": ".NET", "asp.net": ".NET",
    "node.js": "Node.js", "nodejs": "Node.js", "node js": "Node.js",
    "laravel": "Laravel",
    # Clouds
    "aws": "AWS", "amazon web services": "AWS",
    "gcp": "GCP", "google cloud": "GCP", "google cloud platform": "GCP",
    "azure": "Azure", "microsoft azure": "Azure",
    # DBs
    "postgresql": "PostgreSQL", "postgres": "PostgreSQL",
    "mysql": "MySQL",
    "mongodb": "MongoDB", "mongo": "MongoDB",
    "redis": "Redis",
    "elasticsearch": "Elasticsearch", "elastic": "Elasticsearch",
    "cassandra": "Cassandra",
    # Other tools
    "docker": "Docker",
    "kubernetes": "Kubernetes", "k8s": "Kubernetes",
    "git": "Git",
    "ci/cd": "CI/CD", "cicd": "CI/CD", "gitlab ci": "CI/CD", "github actions": "CI/CD", "jenkins": "CI/CD",
    "kafka": "Kafka", "apache kafka": "Kafka",
    "airflow": "Airflow", "apache airflow": "Airflow",
    "spark": "Spark", "apache spark": "Spark",
    "terraform": "Terraform",
}

In [20]:
def normalize_skill(skill):
    return SKILL_ALIAS.get(skill.lower().strip(), skill.strip())

# multi-word aliases need longest match, so I sorted it by token length in desc
_sorted_aliases = sorted(SKILL_ALIAS.keys(), key=lambda x: len(x.split()), reverse=True)

# build one big regex: firstly try multi-word (_mw), then single-word (_sw) with \b
_mw = [re.escape(k) for k in _sorted_aliases if " " in k or "/" in k]
_sw = [re.escape(k) for k in _sorted_aliases if " " not in k and "/" not in k]
_PATTERN = re.compile(
    r"(?i)(" + "|".join(_mw + [r"\b"+ k + r"\b" for k in _sw]) + r")"
)

In [21]:
def extract_skills(text):
    matches = _PATTERN.findall(text)
    seen = set()
    result = []
    for m in matches:
        normalized_skill = normalize_skill(m)
        if normalized_skill not in seen:
            seen.add(normalized_skill)
            result.append(normalized_skill)
    return result

df_cleaned["skills_extracted"] = df_cleaned["description"].apply(extract_skills)

In [22]:
print(f"Some skill extracted rows:")
print(df_cleaned["skills_extracted"].head(3))
print(f"\nRows with >= 1 skill: {(df_cleaned['skills_extracted'].str.len() > 0).sum()}")

Some skill extracted rows:
0             [Python]
2    [FastAPI, Python]
3              [CI/CD]
Name: skills_extracted, dtype: object

Rows with >= 1 skill: 715


In [23]:
# I forgot to mention reasons why in the first notebook I created has_usd_in_description flag, so reason 1 was
# USD was the second most popular currency after KZT,
# so I flagged rows that had USD signs in the description, reason 2:
# these USD mentions might be a useful signal for the neural network
print("has_usd_in_description distribution:")
print(df_cleaned["has_usd_in_description"].value_counts())

has_usd_in_description distribution:
has_usd_in_description
False    1137
True       39
Name: count, dtype: int64


### Some vizualizations before going to next steps (they will go to my streamlit app)

In [24]:
# canonical to category (to color charts)
SKILL_CATEGORY = {
    "Python": "language", "JavaScript": "language", "TypeScript": "language",
    "Java": "language", "Go": "language", "Kotlin": "language", "Swift": "language",
    "C++": "language", "C#": "language", "PHP": "language", "Ruby": "language", "Scala": "language",
    "Django": "framework", "FastAPI": "framework", "Flask": "framework",
    "React": "framework", "Vue": "framework", "Angular": "framework",
    "Spring": "framework", ".NET": "framework", "Node.js": "framework", "Laravel": "framework",
    "AWS": "cloud", "GCP": "cloud", "Azure": "cloud",
    "PostgreSQL": "db", "MySQL": "db", "MongoDB": "db",
    "Redis": "db", "Elasticsearch": "db", "Cassandra": "db",
    "Docker": "tool", "Kubernetes": "tool", "Git": "tool",
    "CI/CD": "tool", "Kafka": "tool", "Airflow": "tool", "Spark": "tool", "Terraform": "tool",
}

CATEGORY_COLOR = {
    "language": "#7357FD",
    "framework": "#02BE9E",
    "cloud": "#FD6363",
    "db": "#F8C85A",
    "tool": "#49BFF6",
}

OUTPUT_DIR = "../data/vizualizations/"
_dark = dict(template="plotly_dark", width=1000, height=500)

In [25]:
# let's see what skills dominate in Kazakhstan IT sphere
all_skills = list(chain.from_iterable(df_cleaned["skills_extracted"]))
skill_counts = Counter(all_skills)
top20 = pd.DataFrame(skill_counts.most_common(20), columns=["skill", "count"])
top20["category"] = top20["skill"].map(SKILL_CATEGORY).fillna("other")
top20["color"] = top20["category"].map(CATEGORY_COLOR).fillna("#AAAAAA")
top20 = top20.sort_values("count")

max_count = top20["count"].max()
top_skill = top20["skill"].iloc[-1]

fig1 = px.bar(
    top20,
    x="count", y="skill",
    orientation="h",
    color="category",
    color_discrete_map=CATEGORY_COLOR,
    labels={"count": "Vacancy count", "skill": ""},
    title="Top-20 in-demand skills - hh.kz",
    **_dark)
fig1.update_layout(legend_title_text="Category", yaxis={"categoryorder": "total ascending"})
fig1.add_annotation(
    x=max_count, y=top_skill,
    text=f"#1 skill appears in {max_count} vacancies"
         f"({max_count/len(df_cleaned)*100:.0f}% of all postings)"
)
fig1.write_html(f"{OUTPUT_DIR}/viz_1.html")
fig1.write_image(f"{OUTPUT_DIR}/viz_1.png")
fig1.show()

In [26]:
# do different roles actually use different stacks or is it all just python everywhere
top_roles = df_cleaned["role"].value_counts().head(10).index.tolist()
top_skills = [skill for skill, _ in skill_counts.most_common(15)]

role_skill_counts = (
    df_cleaned[df_cleaned["role"].isin(top_roles)]
    .explode("skills_extracted")
)

role_skill_counts = role_skill_counts[role_skill_counts["skills_extracted"].isin(top_skills)]

pivot = (
    role_skill_counts.groupby(["role", "skills_extracted"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=top_skills, fill_value=0)
)

fig2 = px.imshow(
    pivot,
    text_auto=True, aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Skill", "y": "Role", "color": "Count"},
    title="Skill demand by Specialization",
    **_dark,
)

fig2.update_layout(xaxis_tickangle=-30, margin=dict(b=90))

max_role, max_skill = pivot.stack().idxmax()
max_cell = pivot.loc[max_role, max_skill]

fig2.add_annotation(
    text=f"Hottest cell: {max_role}×{max_skill}={max_cell}",
    xref="paper",
    yref="paper",
    x=0.01,
    y=-0.20,
    showarrow=False,
    font=dict(size=11)
)

fig2.write_html(f"{OUTPUT_DIR}/viz_2.html")
fig2.write_image(f"{OUTPUT_DIR}/viz_2.png")
fig2.show()

In [27]:
# already know Almaty/Astana dominate but want to see by how much
city_counts = df_cleaned["area"].value_counts().head(15).reset_index()
city_counts.columns = ["city", "count"]
top2_share = city_counts["count"].head(2).sum()/len(df_cleaned)*100

fig3 = px.bar(
    city_counts,
    x="city", y="count",
    labels={"count": "Vacancy count", "city": "City"},
    title="Vacancy Count by City",
    color="count",
    color_continuous_scale="Blues",
    **_dark,
)
fig3.update_layout(coloraxis_showscale=False,    # obvious without this legend
                    xaxis_tickangle=-20)
fig3.add_annotation(
    xref="paper", yref="paper", x=0.98, y=0.95,
    text=f"Top-2 cities = {top2_share:.0f}% of all vacancies",
    showarrow=False, font=dict(size=12)
)
fig3.write_html(f"{OUTPUT_DIR}/viz_3.html")
fig3.write_image(f"{OUTPUT_DIR}/viz_3.png")
fig3.show()

In [28]:
# remote and non-remote workformats
emp_counts = (
    df_cleaned["employment"]
    .value_counts()
    .rename_axis("type")
    .reset_index(name="count")
)

wf_exploded = df_cleaned.explode("work_format")

wf_counts = (
    wf_exploded["work_format"]
    .value_counts()
    .rename_axis("format")
    .reset_index(name="count")
)

remote_count = wf_counts.loc[wf_counts["format"] == "Удалённая работа", "count"].sum()
remote_pct = remote_count / wf_counts["count"].sum()*100

fig4 = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
    subplot_titles=["Employment type", "Work format"],
)

fig4.add_trace(
    go.Pie(labels=emp_counts["type"], values=emp_counts["count"], hole=0.35, name="Employment", textposition="inside",
           insidetextorientation="radial"),
    row=1, col=1,
)
fig4.add_trace(
    go.Pie(labels=wf_counts["format"], values=wf_counts["count"], hole=0.35, name="Work Format"),
    row=1, col=2,
)

fig4.update_layout(
    title_text="Employment type against work format",
    font=dict(size=13),
    **_dark,
)

fig4.add_annotation(
    text=f"Remote share: {remote_pct:.1f}% of work-format mentions",
    x=0.5, y=-0.1,
    showarrow=False,
    font=dict(size=11)
)

fig4.write_html(f"{OUTPUT_DIR}/viz_4.html")
fig4.write_image(f"{OUTPUT_DIR}/viz_4.png")
fig4.show()

In [29]:
# QHow is salary distributed among labeled vacancies? Are there outliers?
sal = df_labeled["salary_mid"]
p5, q1, q3, p95 = sal.quantile([0.05, 0.25, 0.75, 0.95])
med = sal.median()
iqr = q3 -q1

outliers = df_labeled[sal > q3 + 1.5*iqr]
outlier_pct = len(outliers)/len(df_labeled)*100
print(f"Outliers (> Q3+1.5·IQR): {len(outliers)} rows ({outlier_pct:.1f}%)")

fig5 = px.histogram(
    df_labeled,
    x="salary_mid",
    nbins=50,
    labels={"salary_mid": "Salary (KZT)", "count": "Count"},
    title="Salary distribution (KZT) - on labeled vacancies",
    **_dark,
)

for val, label, color in [
    (p5, "P5", "#FF6B6B"),
    (q1, "P25", "#FFD166"),
    (med, "Median", "#00C9A7"),
    (q3, "P75", "#FFD166"),
    (p95, "P95", "#FF6B6B"),
]:
    fig5.add_vline(
        x=val, line_dash="dash", line_color=color, line_width=1.5,
        annotation_text=f"{label}<br>{val/1_000:.0f}k",
        annotation_position="top",
        annotation_font_size=10,
    )

fig5.add_annotation(
    text=f"Median salary: {med/1_000:.0f}K KZT and outliers: {len(outliers)} ({outlier_pct:.1f}%)",
    xref="paper", yref="paper",
    x=0.98, y=0.95,
    xanchor="right",
    showarrow=False,
    font=dict(size=11)
)

fig5.write_html(f"{OUTPUT_DIR}/viz_5.html")
fig5.write_image(f"{OUTPUT_DIR}/viz_5.png")
fig5.show()

Outliers (> Q3+1.5·IQR): 31 rows (8.4%)


In [30]:
# does 6+ yrs actually pay more or is it flat after a point
EXP_LABELS = {
    "noExperience": "No Exp",
    "between1And3": "1–3 yrs",
    "between3And6": "3–6 yrs",
    "moreThan6": "6+ yrs",
}
EXP_ORDER = ["No Exp", "1–3 yrs", "3–6 yrs", "6+ yrs"]

df_labeled_viz = df_labeled.copy()
df_labeled_viz["exp_label"] = df_labeled_viz["experience"].map(EXP_LABELS).fillna("Unknown")
df_labeled_viz = df_labeled_viz[df_labeled_viz["exp_label"].isin(EXP_ORDER)]

fig6 = px.box(
    df_labeled_viz, x="exp_label", y="salary_mid",
    category_orders={"exp_label": EXP_ORDER},
    points="outliers",
    labels={"exp_label": "Experience", "salary_mid": "Salary"},
    title="Salary range by experience grade",
    **_dark,
)
medians = df_labeled_viz.groupby("exp_label")["salary_mid"].median()
junior_med = medians.get("No Exp", 0)
senior_med = medians.get("6+ yrs", 0)
uplift = (senior_med / junior_med - 1) * 100 if junior_med else 0
fig6.add_annotation(
    xref="paper", yref="paper", x=0.98, y=0.95,
    text=f"6+ yrs earns ~{uplift:.0f}% more than No Exp",
    showarrow=False, font=dict(size=11), xanchor="right",
)
fig6.write_html(f"{OUTPUT_DIR}/viz_6.html")
fig6.write_image(f"{OUTPUT_DIR}/viz_6.png")
fig6.show()

In [31]:
# final sanity check before moving on
top3_cities = df_cleaned["area"].value_counts().head(3).index.tolist()
top3_skills = [s for s, _ in skill_counts.most_common(3)]
median_salary = df_labeled["salary_mid"].median()

print(
    f"Total vacancies: {len(df_cleaned)}\n"
    f"Labeled: {len(df_labeled)} ({len(df_labeled)/len(df_cleaned)*100:.1f}%)\n"
    f"Top-3 cities: {', '.join(top3_cities)}\n"
    f"Top-3 skills: {', '.join(top3_skills)}\n"
    f"Median salary: {median_salary:_.0f} KZT"
)

Total vacancies: 1176
Labeled: 367 (31.2%)
Top-3 cities: Алматы, Астана, Караганда
Top-3 skills: Python, CI/CD, Docker
Median salary: 520_443 KZT
